# Relacion entre atributos con UCI Student Performance

Este notebook descarga el conjunto **Student Performance** desde `ucimlrepo` y prueba las funciones de `src.preprocessing`.

Se estudian:
- correlaciones Pearson y Spearman entre variables numericas;
- asociaciones Cramer entre variables categoricas;
- relacion de las variables con objetivos numerico y categorico;
- pares de variables con asociaciones fuertes o aparentemente debiles;
- graficos configurables y matrices reutilizables.

In [ ]:
# Ejecutar una vez si faltan dependencias en el entorno del notebook.
%pip install ucimlrepo pandas numpy matplotlib seaborn scipy

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from ucimlrepo import fetch_ucirepo

# Permite importar src aunque el notebook se ejecute desde otra carpeta.
project_root = Path.cwd()
if not (project_root / 'src').exists():
    project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.preprocessing import (
    cramers_v,
    get_boundaries_iqr,
    get_categorical_columns,
    get_categorical_target_associations,
    get_cramers_v_matrix,
    get_numeric_columns,
    get_numeric_correlation_matrix,
    get_numeric_target_correlations,
    get_top_numeric_correlations,
    plot_categorical_countplots,
    plot_correlation_heatmap,
    plot_cramers_v_heatmap,
    plot_numeric_boxplots,
    plot_numeric_boxplots_by_target,
    plot_numeric_histograms,
)

pd.set_option('display.max_columns', 100)

## 1. Descargar y preparar los datos

In [ ]:
student_performance = fetch_ucirepo(id=320)
df = student_performance.data.features.copy()
df['G3'] = student_performance.data.targets['G3']

# Objetivo categorico derivado: aprobado si la calificacion final es al menos 10.
df['passed'] = (df['G3'] >= 10).astype('category')

print(f'Filas: {df.shape[0]}, columnas: {df.shape[1]}')
display(df.head())
display(df.dtypes.to_frame('dtype'))

## 2. Identificar tipos y revisar outliers

In [ ]:
numeric_columns = get_numeric_columns(df)
categorical_columns = get_categorical_columns(df)

print('Columnas numericas:', numeric_columns)
print('Columnas categoricas:', categorical_columns)

iqr_boundaries = {
    column: get_boundaries_iqr(df[column])
    for column in numeric_columns
}
display(pd.DataFrame(iqr_boundaries, index=['lower_bound', 'upper_bound']).T)

## 3. Correlaciones numericas

Las matrices se conservan en variables para poder filtrarlas, exportarlas o usarlas en otros analisis.

In [ ]:
pearson_matrix = get_numeric_correlation_matrix(df, numeric_columns, method='pearson')
spearman_matrix = get_numeric_correlation_matrix(df, numeric_columns, method='spearman')

print('Matriz Pearson')
display(pearson_matrix.round(3))
print('Matriz Spearman')
display(spearman_matrix.round(3))

top_pearson = get_top_numeric_correlations(pearson_matrix, threshold=0.30, top_n=15)
top_spearman = get_top_numeric_correlations(spearman_matrix, threshold=0.30, top_n=15)
print('Relaciones numericas mas relevantes - Pearson')
display(top_pearson)
print('Relaciones numericas mas relevantes - Spearman')
display(top_spearman)

In [ ]:
# Comparar la relacion de los atributos con el objetivo numerico G3.
g3_pearson = get_numeric_target_correlations(df, target='G3', method='pearson')
g3_spearman = get_numeric_target_correlations(df, target='G3', method='spearman')

display(pd.concat({'pearson': g3_pearson, 'spearman': g3_spearman}, axis=1).round(3))

# Pares aparentemente independientes segun Pearson: revisar siempre con graficos.
weak_pairs = get_top_numeric_correlations(pearson_matrix, threshold=0.0)
weak_pairs = weak_pairs[weak_pairs['absolute_correlation'] < 0.10]
display(weak_pairs)

## 4. Asociaciones categoricas

Cramer V mide la intensidad de la asociacion, no su direccion.

In [ ]:
categorical_analysis_columns = categorical_columns[:8]
cramers_matrix = get_cramers_v_matrix(
    df,
    categorical_columns=categorical_analysis_columns,
    max_vars=None
)

print('Matriz Cramer V')
display(cramers_matrix.round(3))

categorical_target_associations = get_categorical_target_associations(
    df,
    target='passed',
    categorical_columns=categorical_columns
)
print('Asociacion con el objetivo categorico passed')
display(categorical_target_associations.round(3))

print('Ejemplo de Cramer V entre dos columnas')
print(cramers_v(df['school'], df['passed']))

## 5. Graficos configurables

Se usan pocas columnas en algunos graficos para que el notebook sea rapido de inspeccionar. Los parametros `annot`, `cbar`, `fmt` y las mascaras se pueden cambiar.

In [ ]:
plot_numeric_histograms(df, numeric_columns[:6], fig_per_row=3, bins=20, kde=True)
plot_numeric_boxplots(df, numeric_columns[:6], fig_per_row=3)
plot_categorical_countplots(df, categorical_columns[:4], fig_per_row=2, top_n=8)

In [ ]:
plot_numeric_boxplots_by_target(
    df,
    target='passed',
    numeric_columns=['G3', 'absences'],
    fig_per_row=2
)

plot_correlation_heatmap(
    pearson_matrix,
    figsize=(12, 10),
    annot=False,
    cbar=True,
    mask_upper_triangle=True
)

plot_correlation_heatmap(
    spearman_matrix,
    figsize=(12, 10),
    annot=False,
    cbar=False,
    mask_upper_triangle=True
)

plot_cramers_v_heatmap(
    cramers_matrix,
    figsize=(10, 8),
    annot=True,
    cbar=True,
    lower_triangle_only=True
)

## 6. Exportar matrices para continuar el analisis

Estas lineas son opcionales. Las matrices ya estan disponibles en memoria y tambien se pueden guardar como CSV.

In [ ]:
# pearson_matrix.to_csv('data/pearson_matrix.csv')
# spearman_matrix.to_csv('data/spearman_matrix.csv')
# cramers_matrix.to_csv('data/cramers_v_matrix.csv')
print('Matrices exportadas correctamente.')